In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")

input_root = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth"
save_emb_path = "era5_cnn_512_trained.npy"
save_model_path = "weather_autoencoder.pth"

surface_vars = ['2m_temperature', 'mean_sea_level_pressure']
level_vars = ['temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind', 'geopotential']
levels = [1000, 850, 500, 250]

ARCHITECTURE DEFINITION

In [ ]:
class WeatherCNNEncoder(nn.Module):
    def __init__(self, in_channels=22, out_dim=512):
        super(WeatherCNNEncoder, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), 
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), 
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, out_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x

class WeatherCNNDecoder(nn.Module):
    def __init__(self, in_dim=512, out_channels=22):
        super(WeatherCNNDecoder, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.ReLU()
        )
        self.decoder_features = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2), # Output: [128, 4, 4]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # Output: [64, 8, 8]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, out_channels, kernel_size=3, padding=1)  # Output: [22, 8, 8]
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 256, 2, 2)
        x = self.decoder_features(x)
        return x

class WeatherAutoencoder(nn.Module):
    def __init__(self):
        super(WeatherAutoencoder, self).__init__()
        self.encoder = WeatherCNNEncoder(in_channels=22, out_dim=512)
        self.decoder = WeatherCNNDecoder(in_dim=512, out_channels=22)

    def forward(self, x):
        embedding = self.encoder(x)
        reconstructed = self.decoder(embedding)
        return reconstructed, embedding

DATA PROCESSING

In [ ]:
def process_to_cube(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    
    channels = []

    # Surface Data
    surf_df = df[df['level'] == 1000].sort_values(['latitude', 'longitude'], ascending=[False, True])
    if len(surf_df) > 64: surf_df = surf_df.iloc[:64]
        
    for v in surface_vars:
        if v in surf_df.columns:
            grid = surf_df[v].values.reshape(8, 8)
        else:
            fallback = 'temperature' if 'temp' in v else 'geopotential'
            grid = surf_df[fallback].values.reshape(8, 8)
        
        grid = (grid - grid.mean()) / (grid.std() + 1e-6)
        channels.append(grid)

    # Pressure Level Data
    for lvl in levels:
        lvl_df = df[df['level'] == lvl].sort_values(['latitude', 'longitude'], ascending=[False, True])
        if len(lvl_df) > 64: lvl_df = lvl_df.iloc[:64]
            
        for v in level_vars:
            grid = lvl_df[v].values.reshape(8, 8)
            grid = (grid - grid.mean()) / (grid.std() + 1e-6)
            channels.append(grid)
            
    return torch.from_numpy(np.stack(channels)).float().to(device)

PREPARE DATASET

In [ ]:
folders = [f for f in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, f))]
print(f"\n Loading {len(folders)} ERA5 events into memory for training...")

all_tensors = []
valid_folders = []

for folder_name in tqdm(folders):
    csv_path = os.path.join(input_root, folder_name, "center_ERA5_truth.csv")
    if os.path.exists(csv_path):
        try:
            x = process_to_cube(csv_path)
            all_tensors.append(x)
            valid_folders.append(folder_name)
        except Exception as e:
            pass

dataset_tensor = torch.stack(all_tensors) 
dataloader = DataLoader(TensorDataset(dataset_tensor), batch_size=32, shuffle=True)

TRAINING LOOP

In [ ]:
autoencoder = WeatherAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)
epochs = 50

print(f"\n Starting training for {epochs} epochs...")
autoencoder.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in dataloader:
        batch_data = batch[0] 
        
        optimizer.zero_grad()
        reconstruction, _ = autoencoder(batch_data)
        
        loss = criterion(reconstruction, batch_data)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:02d}/{epochs}], MSE Loss: {avg_loss:.4f}")

torch.save(autoencoder.state_dict(), save_model_path)
print(f" Trained model saved to: {save_model_path}")


EXTRACT EMBEDDINGS

In [ ]:
print("\n Extracting learned embeddings for ERA5 data...")
autoencoder.eval()
all_embeddings = {}

with torch.inference_mode():
    for i, folder_name in enumerate(tqdm(valid_folders)):
        x = dataset_tensor[i].unsqueeze(0) # Add batch dimension back: [1, 22, 8, 8]
        
        embedding = autoencoder.encoder(x).cpu().numpy().flatten()
        all_embeddings[folder_name] = embedding

np.save(save_emb_path, all_embeddings)
print(f"\n SUCCESS!")
print(f"Total 512-dim Trained Embeddings saved: {len(all_embeddings)}")
print(f"File path: {os.path.abspath(save_emb_path)}")

Check the extracted embeddings (ERA5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import os

emb_path = "/home/teoaivalis/floods_kg/reanalysis_data/era5_cnn_512_trained.npy"
print(f"Loading embeddings from {emb_path}...")

try:
    era5_dict = np.load(emb_path, allow_pickle=True).item()
except FileNotFoundError:
    print(f"Error: Could not find {emb_path}")
    exit()

event_names = list(era5_dict.keys())
era5_vecs = np.array(list(era5_dict.values()))

print(f"Loaded {len(event_names)} events.")
print(f"Vector shape: {era5_vecs.shape} (Should be N, 512)")

print("\n--- TEST 1: Cross-Similarity (Diversity Check) ---")
similarity_matrix = cosine_similarity(era5_vecs)

np.fill_diagonal(similarity_matrix, np.nan)

mean_sim = np.nanmean(similarity_matrix)
std_sim = np.nanstd(similarity_matrix)

print(f"Average similarity between different events: {mean_sim:.4f}")
print(f"Standard deviation of similarity: {std_sim:.4f}")


print("\n--- TEST 2: Finding the most similar historical events ---")
similarity_matrix = np.nan_to_num(similarity_matrix, nan=-1.0)

max_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
event_A = event_names[max_idx[0]]
event_B = event_names[max_idx[1]]
highest_sim = similarity_matrix[max_idx]

print(f"The two most structurally identical events in your dataset are:")
print(f"1. {event_A}")
print(f"2. {event_B}")
print(f"Cosine Similarity: {highest_sim:.4f}")

print("\n--- TEST 3: Generating 2D PCA Visualization ---")
pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(era5_vecs)

variance_explained = sum(pca.explained_variance_ratio_) * 100
print(f"The top 2 dimensions explain {variance_explained:.2f}% of the variance.")

plt.figure(figsize=(10, 8))
plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], alpha=0.6, edgecolors='w', s=50, c='teal')
plt.title('2D PCA of 512-Dim Weather Embeddings\n(Looking for a structured spread, not a single tight dot)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, alpha=0.3)

plot_path = "era5_trained_embeddings_pca.png"
plt.savefig(plot_path)

LOAD THE TRAINED MODEL

In [ ]:
print("\n Loading trained weights...")
model = WeatherAutoencoder().to(device)

try:
    model.load_state_dict(torch.load(model_weights_path, map_location=device))
    print(" Weights loaded successfully!")
except FileNotFoundError:
    print(f" Error: Could not find '{model_weights_path}'. Make sure it's in the same folder as this script!")
    exit()

model.eval() 

EXTRACTION LOOP

In [ ]:
pangu_folders = [f for f in os.listdir(pangu_root) if os.path.isdir(os.path.join(pangu_root, f))]
all_pangu_embeddings = {}

print(f"\nExtracting Pangu embeddings for {len(pangu_folders)} events...")

with torch.inference_mode():
    for folder_name in tqdm(pangu_folders):
        csv_path = os.path.join(pangu_root, folder_name, "center_Pangu_pred.csv")
        
        if os.path.exists(csv_path):
            try:
                x = process_to_cube(csv_path).unsqueeze(0) 
                
                embedding = model.encoder(x).cpu().numpy().flatten()
                all_pangu_embeddings[folder_name] = embedding
            except Exception as e:
                print(f"\nSkipping {folder_name}: {e}")


SAVE Pangu Embeddings

In [ ]:
np.save(save_pangu_path, all_pangu_embeddings)
print(f"\nSUCCESS!")
print(f"Total Pangu Embeddings saved: {len(all_pangu_embeddings)}")
print(f"File path: {os.path.abspath(save_pangu_path)}")

Check Pangu Embeddings

LOAD THE PANGU EMBEDDINGS

In [ ]:
emb_path = "pangu_cnn_512_trained.npy"
print(f"Loading Pangu embeddings from {emb_path}...")

try:
    pangu_dict = np.load(emb_path, allow_pickle=True).item()
except FileNotFoundError:
    print(f" Error: Could not find {emb_path}")
    exit()

event_names = list(pangu_dict.keys())
pangu_vecs = np.array(list(pangu_dict.values()))

print(f" Loaded {len(event_names)} Pangu prediction events.")
print(f"Vector shape: {pangu_vecs.shape} (Should be N, 512)")

In [ ]:
print("\n--- TEST 1: Cross-Similarity (Internal Diversity) ---")
similarity_matrix = cosine_similarity(pangu_vecs)

np.fill_diagonal(similarity_matrix, np.nan)

mean_sim = np.nanmean(similarity_matrix)
std_sim = np.nanstd(similarity_matrix)

print(f"Average similarity between different Pangu predictions: {mean_sim:.4f}")
print(f"Standard deviation of similarity: {std_sim:.4f}")

print("\n--- TEST 2: Generating 2D PCA Visualization ---")
pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(pangu_vecs)

variance_explained = sum(pca.explained_variance_ratio_) * 100
print(f"The top 2 dimensions explain {variance_explained:.2f}% of the variance.")

plt.figure(figsize=(10, 8))
plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], alpha=0.7, edgecolors='w', s=50, c='coral')
plt.title('2D PCA of 512-Dim Pangu Prediction Embeddings\n(Looking for a structured spread)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, alpha=0.3)

# Save the plot
plot_path = "pangu_trained_embeddings_pca.png"
plt.savefig(plot_path)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pangu_root = "/home/teoaivalis/floods_kg/reanalysis_data/pangu_predictions"
save_pangu_path = "pangu_cnn_512_trained_v2.npy"
model_weights_path = "/home/teoaivalis/floods_kg/weather_autoencoder_3d_multi_day.pth"

surface_vars = ['2m_temperature', 'mean_sea_level_pressure']
level_vars = ['temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind', 'geopotential']
levels = [1000, 850, 500, 250]

class ResBlock3D(nn.Module):
    def __init__(self, channels):
        super(ResBlock3D, self).__init__()
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(channels)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(channels)
        
    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class WeatherEncoder3D(nn.Module):
    def __init__(self, in_channels=7, out_dim=512):
        super(WeatherEncoder3D, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU()
        )
        self.res1 = ResBlock3D(32)
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2))
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.res2 = ResBlock3D(64)
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2))
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2 * 2, out_dim)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.res1(x))
        x = self.pool2(self.res2(self.conv2(x)))
        x = self.fc(x)
        return x

class WeatherDecoder3D(nn.Module):
    def __init__(self, in_dim=512, out_channels=7):
        super(WeatherDecoder3D, self).__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, 512), nn.ReLU())
        self.up1 = nn.ConvTranspose3d(64, 32, kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.res3 = ResBlock3D(32)
        self.up2 = nn.ConvTranspose3d(32, 16, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.res4 = ResBlock3D(16)
        self.out_conv = nn.Conv3d(16, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 64, 2, 2, 2)
        x = self.res3(self.up1(x))
        x = self.res4(self.up2(x))
        x = self.out_conv(x)
        return x

class WeatherAutoencoder3D(nn.Module):
    def __init__(self):
        super(WeatherAutoencoder3D, self).__init__()
        self.encoder = WeatherEncoder3D()
        self.decoder = WeatherDecoder3D()

    def forward(self, x):
        return self.decoder(self.encoder(x)), self.encoder(x)

def process_to_3d_cube(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    
    level_tensors = []
    for v in level_vars:
        var_levels = []
        for lvl in levels: 
            lvl_df = df[df['level'] == lvl].sort_values(['latitude', 'longitude'], ascending=[False, True])
            if len(lvl_df) > 64: lvl_df = lvl_df.iloc[:64]
            grid = lvl_df[v].values.reshape(8, 8)
            grid = (grid - grid.mean()) / (grid.std() + 1e-6)
            var_levels.append(grid)
        level_tensors.append(np.stack(var_levels))
    level_tensors = np.stack(level_tensors)
    
    surf_df = df[df['level'] == 1000].sort_values(['latitude', 'longitude'], ascending=[False, True])
    if len(surf_df) > 64: surf_df = surf_df.iloc[:64]
    
    surf_tensors = []
    for v in surface_vars:
        if v in surf_df.columns:
            grid = surf_df[v].values.reshape(8, 8)
        else:
            fallback = 'temperature' if 'temp' in v else 'geopotential'
            grid = surf_df[fallback].values.reshape(8, 8)
            
        grid = (grid - grid.mean()) / (grid.std() + 1e-6)
        grid_3d = np.repeat(grid[np.newaxis, :, :], 4, axis=0) 
        surf_tensors.append(grid_3d)
    surf_tensors = np.stack(surf_tensors)
    
    cube = np.concatenate([level_tensors, surf_tensors], axis=0)
    return torch.from_numpy(cube).float().to(device)


model = WeatherAutoencoder3D().to(device)
model.eval() 

pangu_folders = [f for f in os.listdir(pangu_root) if os.path.isdir(os.path.join(pangu_root, f))]
all_pangu_embeddings = {}

print(f"\nExtracting Pangu embeddings for {len(pangu_folders)} events...")

with torch.inference_mode():
    for folder_name in tqdm(pangu_folders):
        csv_path = os.path.join(pangu_root, folder_name, "center_Pangu_pred.csv")
        
        if os.path.exists(csv_path):
            try:
                x = process_to_3d_cube(csv_path).unsqueeze(0) 
                embedding = model.encoder(x).cpu().numpy().flatten()
                all_pangu_embeddings[folder_name] = embedding
            except Exception as e:
                print(f"\nSkipping {folder_name}: {e}")

np.save(save_pangu_path, all_pangu_embeddings)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

try:
    era5_dict = np.load("era5_cnn_512_event_day_only.npy", allow_pickle=True).item()
    pangu_dict = np.load("pangu_cnn_512_trained_v2.npy", allow_pickle=True).item()
except FileNotFoundError as e:
    exit()

common_events = list(set(era5_dict.keys()).intersection(set(pangu_dict.keys())))
print(f"Found {len(common_events)} matching events to compare.\n")

era5_vecs = np.array([era5_dict[e] for e in common_events])
pangu_vecs = np.array([pangu_dict[e] for e in common_events])


print("--- USE CASE 1: Pangu 3D Prediction Accuracy ---")

accuracy_scores = cosine_similarity(pangu_vecs, era5_vecs).diagonal()

mean_accuracy = np.mean(accuracy_scores)
best_idx = np.argmax(accuracy_scores)
worst_idx = np.argmin(accuracy_scores)

print(f"Average Prediction Accuracy (Cosine Sim): {mean_accuracy:.4f}")
print(f"Best Predicted Event:  {common_events[best_idx]} (Score: {accuracy_scores[best_idx]:.4f})")
print(f"Worst Predicted Event: {common_events[worst_idx]} (Score: {accuracy_scores[worst_idx]:.4f})\n")

plt.figure(figsize=(8, 5))
plt.hist(accuracy_scores, bins=20, color='royalblue', edgecolor='black', alpha=0.8)
plt.axvline(mean_accuracy, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_accuracy:.2f}')
plt.title('Pangu-Weather Prediction Accuracies (3D Structural Similarity)')
plt.xlabel('Cosine Similarity to ERA5 Ground Truth')
plt.ylabel('Number of Events')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('pangu_v2_accuracy_histogram.png')


print("--- USE CASE 2: Structural Event Retrieval ---")

num_tests = 3
test_indices = np.random.choice(range(len(common_events)), size=num_tests, replace=False)

for test_idx in test_indices:
    target_event_name = common_events[test_idx]
    
    target_pangu_vector = pangu_vecs[test_idx].reshape(1, -1) 

    print(f"\nQuery (Pangu Prediction): {target_event_name}")

    search_similarities = cosine_similarity(target_pangu_vector, era5_vecs)[0]

    top_5_indices = np.argsort(search_similarities)[::-1][:5]

    print("Top 5 Most Similar ERA5 Historical Events:")
    for rank, idx in enumerate(top_5_indices):
        event_name = common_events[idx]
        score = search_similarities[idx]